# 实时行人属性识别 (Real-time Pedestrian Attribute Recognition)

这个 Notebook 实现了一个实时系统，结合了 YOLOv8 进行行人检测和 MobileNetV4 进行多标签属性识别。

**功能流程:**
1.  使用 YOLOv8 检测视频流中的行人。
2.  将检测到的行人区域裁剪出来。
3.  调整裁剪图像大小至 **112x112** (Phase2 模型配置)。
4.  使用预训练的 MobileNetV4 模型识别行人的 26 个属性。
5.  在视频流中实时绘制检测框和属性标签。

In [1]:
import cv2
import torch
import torch.nn as nn
import numpy as np
from ultralytics import YOLO
import timm
from torchvision import transforms as T
from PIL import Image
import time

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


/Users/chenshi/Desktop/ipynotebook/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 定义属性列表 (参考 example.txt)
ATTRIBUTES = [
    'Female', 'AgeOver60', 'Age18-60', 'AgeLess18', 
    'Front', 'Side', 'Back', 
    'Hat', 'Glasses', 
    'HandBag', 'ShoulderBag', 'Backpack', 'HoldObjectsInFront', 
    'ShortSleeve', 'LongSleeve', 
    'UpperStride', 'UpperLogo', 'UpperPlaid', 'UpperSplice', 
    'LowerStripe', 'LowerPattern', 'LongCoat', 
    'Trousers', 'Shorts', 'Skirt&Dress', 'boots'
]

print(f"Total attributes: {len(ATTRIBUTES)}")

Total attributes: 26


In [3]:
# 定义 MobileNet 模型结构 (与 phase2.ipynb 训练代码保持一致)
class PedestrianAttributeNet(nn.Module):
    def __init__(self, model_name, num_classes, img_size=112):
        super(PedestrianAttributeNet, self).__init__()
        # 加载预训练 backbone
        self.backbone = timm.create_model(
            model_name, 
            pretrained=False, # 推理时不需要下载预训练权重，因为我们会加载自己的权重
            num_classes=0, 
            global_pool='' 
        )
        
        # 自动获取 backbone 输出特征维度
        with torch.no_grad():
            dummy = torch.randn(1, 3, img_size, img_size)
            features = self.backbone(dummy)
            self.in_features = features.shape[1]
        
        # 自定义分类头 (与 phase2.ipynb 结构一致)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(self.in_features),
            nn.Dropout(0.2),
            nn.Linear(self.in_features, num_classes)
        )
        
    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)

# 配置参数 (与 phase2.ipynb 对齐)
IMG_SIZE = 112  # Phase2 训练使用的输入尺寸
model_name = 'mobilenetv4_conv_small_050.e3000_r224_in1k'
num_classes = 26

# 初始化模型
attr_model = PedestrianAttributeNet(model_name, num_classes, img_size=IMG_SIZE)

# 加载权重 (更新为 phase2 训练的模型路径)
weights_path = '/Users/chenshi/Desktop/ipynotebook/models/mobilenet/best_model.pth'
try:
    attr_model.load_state_dict(torch.load(weights_path, map_location=device))
    print(f"Successfully loaded weights from {weights_path}")
except FileNotFoundError:
    print(f"Error: Weights file not found at {weights_path}")
except Exception as e:
    print(f"Error loading weights: {e}")

attr_model = attr_model.to(device)
attr_model.eval()
print(f"Model input size: {IMG_SIZE}x{IMG_SIZE}")

Successfully loaded weights from /Users/chenshi/Desktop/ipynotebook/models/mobilenet/best_model.pth
Model input size: 112x112


In [4]:
# 加载 YOLO 模型
try:
    yolo_model = YOLO('yolov8n.pt')
    print("YOLOv8 model loaded successfully.")
except Exception as e:
    print(f"Error loading YOLO model: {e}")

YOLOv8 model loaded successfully.


In [5]:
# 定义预处理转换 (更新为 phase2 的输入尺寸 112x112)
preprocess = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_attributes(image_crop, model, threshold=0.5):
    """
    对裁剪的行人图像进行属性预测
    """
    # 转换为 PIL Image
    if isinstance(image_crop, np.ndarray):
        image_pil = Image.fromarray(cv2.cvtColor(image_crop, cv2.COLOR_BGR2RGB))
    else:
        image_pil = image_crop
        
    # 预处理
    input_tensor = preprocess(image_pil).unsqueeze(0).to(device)
    
    # 推理
    with torch.no_grad():
        logits = model(input_tensor)
        probs = torch.sigmoid(logits)[0].cpu().numpy()
        
    # 获取激活的属性
    active_indices = np.where(probs > threshold)[0]
    active_attributes = [(ATTRIBUTES[i], probs[i]) for i in active_indices]
    
    return active_attributes

In [7]:
# 实时视频处理循环
def run_webcam_inference(source=0, conf_thresh=0.5):
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        # 尝试 source 1
        cap = cv2.VideoCapture(1)
        if not cap.isOpened():
            print("Cannot open camera")
            return

    print("Starting video loop. Press 'q' to exit.")
    
    # 帧率计算
    prev_frame_time = 0
    new_frame_time = 0

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("Failed to grab frame")
                break
                
            # YOLO 检测
            # classes=[0] 仅检测人
            results = yolo_model(frame, classes=[0], verbose=False, conf=conf_thresh)
            
            # 处理检测结果
            for result in results:
                boxes = result.boxes
                for box in boxes:
                    # 获取边界框坐标
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                    
                    # 确保坐标在图像范围内
                    h, w, _ = frame.shape
                    x1 = max(0, x1)
                    y1 = max(0, y1)
                    x2 = min(w, x2)
                    y2 = min(h, y2)
                    
                    # 裁剪行人图像
                    if x2 > x1 and y2 > y1:
                        person_crop = frame[y1:y2, x1:x2]
                        
                        # 属性识别
                        attrs = predict_attributes(person_crop, attr_model)
                        
                        # 绘制边界框
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        
                        # 绘制属性文本
                        text_y = y1 - 10
                        if text_y < 10: text_y = y1 + 20
                        
                        # 筛选一些主要属性显示，避免文字过多
                        display_attrs = []
                        for name, score in attrs:
                            # 简化显示：只显示性别、年龄、朝向、背包等关键信息
                            # 或者显示所有置信度高的
                            display_attrs.append(name)
                        
                        # 显示所有属性，不省略
                        label_text = ", ".join(display_attrs)
                            
                        cv2.putText(frame, label_text, (x1, text_y), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                        
                        # 在控制台打印详细信息 (可选)
                        # print(f"Person at ({x1},{y1}): {display_attrs}")

            # 计算 FPS
            new_frame_time = time.time()
            fps = 1/(new_frame_time-prev_frame_time)
            prev_frame_time = new_frame_time
            cv2.putText(frame, f"FPS: {int(fps)}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

            cv2.imshow('Pedestrian Attribute Recognition', frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
                
    except KeyboardInterrupt:
        print("Interrupted by user")
    finally:
        cap.release()
        cv2.destroyAllWindows()

# 运行
# 注意：在某些环境中 cv2.imshow 可能无法工作，需要确保本地环境支持 GUI
run_webcam_inference()

Starting video loop. Press 'q' to exit.
Failed to grab frame
Failed to grab frame
